In [9]:
!git clone https://github.com/sahersh17/HT_detection_GNN.git

Cloning into 'HT_detection_GNN'...
remote: Enumerating objects: 3149, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3149 (delta 0), reused 0 (delta 0), pack-reused 3146 (from 1)
Receiving objects: 100% (3149/3149), 181.28 MiB | 25.45 MiB/s, done.
Resolving deltas: 100% (1864/1864), done.
Updating files: 100% (1991/1991), done.


In [10]:
import subprocess
%cd /content/HT_detection_GNN
print(subprocess.run(['git', 'log', '--oneline', '-5'], capture_output=True, text=True).stdout)
print(subprocess.run(['git', 'status'], capture_output=True, text=True).stdout)

/content/HT_detection_GNN
5913a62 removed git personal acess tokens
62cf89f Merge branch 'main' of https://github.com/sahersh17/HT_detection_GNN
0005788 updating graph feature extractor and graph builder
ec41e2a updated semantic feature extractor & embeddings using 512 tokens & sliding window chunks
6780030 verifying semantic coverage

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



In [ ]:
GIT_PAT="your token"

In [13]:
import os
pat = GIT_PAT  # wherever you've stored it
remote_url = f'https://{pat}@github.com/sahersh17/HT_detection_GNN.git'

In [14]:
!git config --global user.email "saher.sharf17@gmail.com"
!git config --global user.name "sahersh17"

In [4]:
!pip install torch-geometric

In [2]:
"""
GNN <-> RTL cross-attention fusion module for hardware Trojan detection.

Pipeline:
    PyG Batch(graphs) -> GNNEncoder -> to_dense_batch -> GNNCodeCrossAttention
    -> masked mean pool -> classifier

Validated against real AES-T100 clean/trojan netlist + RTL chunk samples.
"""
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_dense_batch


class GNNEncoder(nn.Module):
    """Lifts raw node features (gate-class one-hot + degree/controllability/
    observability, 20-dim) into a hidden embedding space via message passing."""

    def __init__(self, in_dim=20, hidden_dim=128):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x


class GNNCodeCrossAttention(nn.Module):
    """Cross-attention fusion: GNN node embeddings attend over RTL token
    embeddings (CodeBERT chunk output). attn_weights doubles as the
    localization/explainability signal (which RTL tokens each node used)."""

    def __init__(self, gnn_dim, code_dim, hidden_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.node_proj = nn.Linear(gnn_dim, hidden_dim)
        self.code_proj = nn.Linear(code_dim, hidden_dim)
        self.cross_attn = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, node_embeds, code_token_embeds, code_attn_mask=None,
                average_attn_weights=True):
        """
        node_embeds:        (B, N, gnn_dim)   from to_dense_batch(GNN output)
        code_token_embeds:  (B, T, code_dim)  padded RTL chunk token embeddings
        code_attn_mask:     (B, T) bool, True = PAD (ignore)
        Returns:
            fused_nodes: (B, N, hidden_dim)   -- mask with node_mask before pooling
            attn_weights: (B, N, T) or (B, num_heads, N, T) if average_attn_weights=False
        """
        q = self.node_proj(node_embeds)
        kv = self.code_proj(code_token_embeds)
        attn_out, attn_weights = self.cross_attn(
            q, kv, kv,
            key_padding_mask=code_attn_mask,
            need_weights=True,
            average_attn_weights=average_attn_weights,
        )
        x = self.norm1(q + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights


def masked_mean_pool(fused_nodes, node_mask):
    """Graph-level embedding from node-level fused output, ignoring padded nodes."""
    mask_f = node_mask.unsqueeze(-1).float()
    return (fused_nodes * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-8)


if __name__ == "__main__":
    # Smoke test with dummy shapes matching real data (20-dim nodes, 768-dim RTL tokens)
    from torch_geometric.data import Data, Batch
    from torch.nn.utils.rnn import pad_sequence

    g1 = Data(x=torch.randn(50, 20), edge_index=torch.randint(0, 50, (2, 80)), y=torch.tensor([0]))
    g2 = Data(x=torch.randn(40, 20), edge_index=torch.randint(0, 40, (2, 60)), y=torch.tensor([1]))
    batch = Batch.from_data_list([g1, g2])

    encoder = GNNEncoder()
    node_embeds = encoder(batch.x, batch.edge_index)
    dense_nodes, node_mask = to_dense_batch(node_embeds, batch.batch)

    rtl = [torch.randn(30, 768), torch.randn(12, 768)]
    rtl_padded = pad_sequence(rtl, batch_first=True)
    code_mask = torch.zeros(rtl_padded.shape[:2], dtype=torch.bool)
    for i, r in enumerate(rtl):
        code_mask[i, r.shape[0]:] = True

    fusion = GNNCodeCrossAttention(gnn_dim=128, code_dim=768)
    fused, attn = fusion(dense_nodes, rtl_padded, code_attn_mask=code_mask)
    graph_embeds = masked_mean_pool(fused, node_mask)

    print("fused_nodes:", fused.shape, "| attn_weights:", attn.shape, "| graph_embeds:", graph_embeds.shape)

fused_nodes: torch.Size([2, 50, 256]) | attn_weights: torch.Size([2, 50, 30]) | graph_embeds: torch.Size([2, 256])


In [10]:
import os
print(os.getcwd())

/content


In [12]:
import sys
sys.path.append('/content/HT_detection_GNN/trusthub/scripts')

In [14]:
"""
Combined GNN <-> RTL cross-attention fusion + training pipeline, GPU-aware.
Paste this as ONE notebook cell -- no cross-file import needed.
"""
import re
import random
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch_geometric.data import Batch
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_dense_batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ---------------------------------------------------------------------------
# Model components
# ---------------------------------------------------------------------------
class GNNEncoder(nn.Module):
    def __init__(self, in_dim=20, hidden_dim=128):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)


class GNNCodeCrossAttention(nn.Module):
    def __init__(self, gnn_dim, code_dim, hidden_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.node_proj = nn.Linear(gnn_dim, hidden_dim)
        self.code_proj = nn.Linear(code_dim, hidden_dim)
        self.cross_attn = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2), nn.GELU(), nn.Linear(hidden_dim * 2, hidden_dim))
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, node_embeds, code_token_embeds, code_attn_mask=None, average_attn_weights=True):
        q = self.node_proj(node_embeds)
        kv = self.code_proj(code_token_embeds)
        attn_out, attn_weights = self.cross_attn(
            q, kv, kv, key_padding_mask=code_attn_mask,
            need_weights=True, average_attn_weights=average_attn_weights,
        )
        x = self.norm1(q + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights


def masked_mean_pool(fused_nodes, node_mask):
    mask_f = node_mask.unsqueeze(-1).float()
    return (fused_nodes * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-8)


class TrojanDetector(nn.Module):
    def __init__(self, node_in_dim=20, gnn_dim=128, code_dim=768, hidden_dim=256, num_classes=2):
        super().__init__()
        self.encoder = GNNEncoder(in_dim=node_in_dim, hidden_dim=gnn_dim)
        self.fusion = GNNCodeCrossAttention(gnn_dim, code_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, pyg_batch, rtl_padded, code_mask):
        node_embeds = self.encoder(pyg_batch.x, pyg_batch.edge_index)
        dense_nodes, node_mask = to_dense_batch(node_embeds, pyg_batch.batch)
        fused, attn_weights = self.fusion(dense_nodes, rtl_padded, code_attn_mask=code_mask)
        graph_embed = masked_mean_pool(fused, node_mask)
        logits = self.classifier(graph_embed)
        return logits, attn_weights


# ---------------------------------------------------------------------------
# Dataset pairing
# ---------------------------------------------------------------------------
def discover_samples(root_dir):
    root = Path(root_dir)
    samples = []
    for netlist_path in sorted(root.glob("*/*/*_netlist.pt")):
        variant_dir = netlist_path.parent
        family = variant_dir.parent.name
        variant = variant_dir.name
        side = netlist_path.name.replace("_netlist.pt", "")

        chunk_paths = sorted(
            variant_dir.glob(f"{side}_rtl_chunk_*.pt"),
            key=lambda p: int(re.search(r"chunk_(\d+)", p.name).group(1)),
        )
        if not chunk_paths:
            print(f"WARNING: no RTL chunks found for {netlist_path}, skipping")
            continue

        samples.append({
            "netlist_path": netlist_path, "rtl_chunk_paths": chunk_paths,
            "family": family, "variant": variant, "side": side,
        })
    return samples


class HWTrojanDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        graph = torch.load(s["netlist_path"], weights_only=False)
        rtl = torch.cat([torch.load(p) for p in s["rtl_chunk_paths"]], dim=0)
        return graph, rtl, graph.y.item()


def collate_fn(batch):
    graphs, rtls, labels = zip(*batch)
    pyg_batch = Batch.from_data_list(list(graphs))
    rtl_padded = pad_sequence(list(rtls), batch_first=True)
    lengths = [r.shape[0] for r in rtls]
    code_mask = torch.zeros(rtl_padded.shape[:2], dtype=torch.bool)
    for i, l in enumerate(lengths):
        code_mask[i, l:] = True
    labels = torch.tensor(labels, dtype=torch.long)
    return pyg_batch, rtl_padded, code_mask, labels


def variant_grouped_split(samples, val_frac=0.2, seed=42):
    groups = defaultdict(list)
    for s in samples:
        groups[(s["family"], s["variant"])].append(s)
    keys = list(groups.keys())
    random.Random(seed).shuffle(keys)
    n_val = max(1, int(len(keys) * val_frac))
    val_keys, train_keys = set(keys[:n_val]), set(keys[n_val:])
    train = [s for k in train_keys for s in groups[k]]
    val = [s for k in val_keys for s in groups[k]]
    return train, val


# ---------------------------------------------------------------------------
# Training loop -- now with .to(device) everywhere it matters
# ---------------------------------------------------------------------------
def train(root_dir, epochs=5, batch_size=2, lr=1e-3, val_frac=0.2, class_weights=None):
    samples = discover_samples(root_dir)
    print(f"Discovered {len(samples)} samples across "
          f"{len(set((s['family'], s['variant']) for s in samples))} variants")

    train_samples, val_samples = variant_grouped_split(samples, val_frac=val_frac)
    print(f"Train: {len(train_samples)} samples | Val: {len(val_samples)} samples")

    train_loader = DataLoader(HWTrojanDataset(train_samples), batch_size=batch_size,
                               shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(HWTrojanDataset(val_samples), batch_size=batch_size,
                             shuffle=False, collate_fn=collate_fn)

    model = TrojanDetector().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if class_weights is not None:
        class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for pyg_batch, rtl_padded, code_mask, labels in train_loader:
            pyg_batch = pyg_batch.to(device)
            rtl_padded = rtl_padded.to(device)
            code_mask = code_mask.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits, _ = model(pyg_batch, rtl_padded, code_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for pyg_batch, rtl_padded, code_mask, labels in val_loader:
                pyg_batch = pyg_batch.to(device)
                rtl_padded = rtl_padded.to(device)
                code_mask = code_mask.to(device)
                labels = labels.to(device)
                logits, _ = model(pyg_batch, rtl_padded, code_mask)
                v_correct += (logits.argmax(dim=1) == labels).sum().item()
                v_total += labels.size(0)
        val_acc = v_correct / v_total if v_total else float("nan")

        print(f"epoch {epoch:2d} | train_loss {train_loss:.4f} | "
              f"train_acc {train_acc:.3f} | val_acc {val_acc:.3f}")

    return model


# Run it (adjust root_dir to your actual dataset path in the repo)
# adjust this path if your dataset folder lives elsewhere under the repo
model = train(root_dir="/content/HT_detection_GNN/trusthub/dataset", epochs=5, batch_size=2)

Using device: cuda
Discovered 0 samples across 0 variants
Train: 0 samples | Val: 0 samples


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [16]:
import os

def print_tree(path, prefix="", max_depth=3, depth=0):
    if depth > max_depth or not os.path.exists(path):
        return
    try:
        entries = sorted(os.listdir(path))
    except PermissionError:
        return
    for entry in entries:
        full = os.path.join(path, entry)
        print(prefix + entry)
        if os.path.isdir(full):
            print_tree(full, prefix + "  ", max_depth, depth + 1)

print_tree("/content/HT_detection_GNN/trusthub/features")

data
  AES
    AES-T100
      clean_netlist.pt
      clean_rtl_embedding.pt
      clean_rtl_tokens.pt
      clean_rtl_tokens_chunk.pt
      trojan_netlist.pt
      trojan_rtl_embedding.pt
      trojan_rtl_tokens.pt
      trojan_rtl_tokens_chunk.pt
    AES-T1000
      clean_netlist.pt
      clean_rtl_embedding.pt
      clean_rtl_tokens.pt
      clean_rtl_tokens_chunk.pt
      trojan_netlist.pt
      trojan_rtl_embedding.pt
      trojan_rtl_tokens.pt
      trojan_rtl_tokens_chunk.pt
    AES-T1100
      clean_netlist.pt
      clean_rtl_embedding.pt
      clean_rtl_tokens.pt
      clean_rtl_tokens_chunk.pt
      trojan_netlist.pt
      trojan_rtl_embedding.pt
      trojan_rtl_tokens.pt
      trojan_rtl_tokens_chunk.pt
    AES-T1200
      clean_netlist.pt
      clean_rtl_embedding.pt
      clean_rtl_tokens.pt
      clean_rtl_tokens_chunk.pt
      trojan_netlist.pt
      trojan_rtl_embedding.pt
      trojan_rtl_tokens.pt
      trojan_rtl_tokens_chunk.pt
    AES-T1300
      clean_netlist.pt
 

In [17]:
"""
Robust dataset discovery for the real trusthub/data layout, which uses
inconsistent naming across families:

  AES-*             : clean_netlist.pt / trojan_netlist.pt, clean|trojan_rtl_tokens_chunk.pt
  RS232-T100..T900  : standard_netlist.pt only (trojan_rtl_* files exist but NO matching
                       trojan netlist -- these RTL files are currently orphaned, skipped)
  RS232-T1000..2000 : 90nm_netlist.pt / 180nm_netlist.pt, only ONE rtl file (trojan_rtl_tokens_chunk.pt)
                       -- ASSUMPTION: both tech-node netlists pair with that same trojan RTL
                       (same source RTL, two synthesis targets). Flagged, not guaranteed.
  RS232-T2100..2400 : tjfree_netlist.pt / tjin_netlist.pt, but RTL files are named
                       clean_rtl_embedding_chunk.pt / trojan_rtl_tokens_chunk.pt (different
                       prefix AND different suffix than the netlist side name)

Labels (y) are read from the netlist .pt file itself, never inferred from filename,
so the 90nm/180nm ambiguity above does not affect label correctness -- only which
RTL embedding (if any) gets attached to a given graph.
"""
import re
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import Dataset

# side name (from *_netlist.pt) -> ordered list of candidate RTL filenames to try
RTL_ALIASES = {
    "clean":    ["clean_rtl_tokens_chunk.pt", "clean_rtl_embedding_chunk.pt"],
    "standard": ["standard_rtl_tokens_chunk.pt", "standard_rtl_embedding_chunk.pt"],
    "trojan":   ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],
    "tjfree":   ["clean_rtl_tokens_chunk.pt", "clean_rtl_embedding_chunk.pt"],
    "tjin":     ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],
    "90nm":     ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],  # ASSUMPTION -- verify
    "180nm":    ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],  # ASSUMPTION -- verify
}


def discover_samples(root_dir, verbose=True):
    root = Path(root_dir)
    samples = []
    skipped = []

    for netlist_path in sorted(root.glob("*/*/*_netlist.pt")):
        variant_dir = netlist_path.parent
        family = variant_dir.parent.name
        variant = variant_dir.name
        side = netlist_path.name.replace("_netlist.pt", "")

        candidates = RTL_ALIASES.get(side, [f"{side}_rtl_tokens_chunk.pt", f"{side}_rtl_embedding_chunk.pt"])
        rtl_path = next((variant_dir / c for c in candidates if (variant_dir / c).exists()), None)

        if rtl_path is None:
            skipped.append((family, variant, side, "no matching RTL file found"))
            continue

        samples.append({
            "netlist_path": netlist_path, "rtl_path": rtl_path,
            "family": family, "variant": variant, "side": side,
        })

    if verbose:
        print(f"Matched {len(samples)} netlist+RTL pairs, skipped {len(skipped)} netlist files with no RTL match")
        if skipped:
            print("\nSkipped (family, variant, side, reason):")
            for row in skipped:
                print("  ", row)
    return samples, skipped


def report_dataset_coverage(root_dir):
    """Run before training: shows per-family sample counts and label balance
    so you can sanity check the data BEFORE burning GPU time on it."""
    samples, skipped = discover_samples(root_dir, verbose=True)

    by_family = defaultdict(lambda: {"total": 0, "y0": 0, "y1": 0})
    for s in samples:
        g = torch.load(s["netlist_path"], map_location="cpu", weights_only=False)
        y = g.y.item()
        by_family[s["family"]]["total"] += 1
        by_family[s["family"]][f"y{y}"] += 1

    print("\nPer-family usable sample counts (y0 = label 0, y1 = label 1):")
    for family, counts in by_family.items():
        print(f"  {family}: total={counts['total']}  y0={counts['y0']}  y1={counts['y1']}")

    total_variants = len(set((s["family"], s["variant"]) for s in samples))
    print(f"\nTotal usable samples: {len(samples)} across {total_variants} variants")
    return samples, skipped


class HWTrojanDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        graph = torch.load(s["netlist_path"], weights_only=False)
        rtl = torch.load(s["rtl_path"])
        return graph, rtl, graph.y.item()


if __name__ == "__main__":
    import sys
    root = sys.argv[1] if len(sys.argv) > 1 else "data"
    report_dataset_coverage(root)

Matched 0 netlist+RTL pairs, skipped 0 netlist files with no RTL match

Per-family usable sample counts (y0 = label 0, y1 = label 1):

Total usable samples: 0 across 0 variants


In [18]:
import os

# 1. Does the path even exist?
root_dir = "/content/HT_detection_GNN/trusthub/data"
print("exists:", os.path.exists(root_dir))

# 2. If it doesn't, search for it
for dirpath, dirnames, filenames in os.walk("/content/HT_detection_GNN"):
    if "manifest.json" in filenames:
        print("Found data root at:", dirpath)

exists: False
Found data root at: /content/HT_detection_GNN/trusthub/features


In [21]:
"""
Robust dataset discovery for the real trusthub/data layout, which uses
inconsistent naming across families:

  AES-*             : clean_netlist.pt / trojan_netlist.pt, clean|trojan_rtl_tokens_chunk.pt
  RS232-T100..T900  : standard_netlist.pt only (trojan_rtl_* files exist but NO matching
                       trojan netlist -- these RTL files are currently orphaned, skipped)
  RS232-T1000..2000 : 90nm_netlist.pt / 180nm_netlist.pt, only ONE rtl file (trojan_rtl_tokens_chunk.pt)
                       -- ASSUMPTION: both tech-node netlists pair with that same trojan RTL
                       (same source RTL, two synthesis targets). Flagged, not guaranteed.
  RS232-T2100..2400 : tjfree_netlist.pt / tjin_netlist.pt, but RTL files are named
                       clean_rtl_embedding_chunk.pt / trojan_rtl_tokens_chunk.pt (different
                       prefix AND different suffix than the netlist side name)

Labels (y) are read from the netlist .pt file itself, never inferred from filename,
so the 90nm/180nm ambiguity above does not affect label correctness -- only which
RTL embedding (if any) gets attached to a given graph.
"""
import re
from pathlib import Path
from collections import defaultdict

import torch
from torch.utils.data import Dataset

# side name (from *_netlist.pt) -> ordered list of candidate RTL filenames to try
RTL_ALIASES = {
    "clean":    ["clean_rtl_tokens_chunk.pt", "clean_rtl_embedding_chunk.pt"],
    "standard": ["standard_rtl_tokens_chunk.pt", "standard_rtl_embedding_chunk.pt"],
    "trojan":   ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],
    "tjfree":   ["clean_rtl_tokens_chunk.pt", "clean_rtl_embedding_chunk.pt"],
    "tjin":     ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],
    "90nm":     ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],  # ASSUMPTION -- verify
    "180nm":    ["trojan_rtl_tokens_chunk.pt", "trojan_rtl_embedding_chunk.pt"],  # ASSUMPTION -- verify
}


def discover_samples(root_dir, verbose=True):
    root = Path(root_dir)
    samples = []
    skipped = []

    for netlist_path in sorted(root.glob("*/*/*_netlist.pt")):
        variant_dir = netlist_path.parent
        family = variant_dir.parent.name
        variant = variant_dir.name
        side = netlist_path.name.replace("_netlist.pt", "")

        candidates = RTL_ALIASES.get(side, [f"{side}_rtl_tokens_chunk.pt", f"{side}_rtl_embedding_chunk.pt"])
        rtl_path = next((variant_dir / c for c in candidates if (variant_dir / c).exists()), None)

        if rtl_path is None:
            skipped.append((family, variant, side, "no matching RTL file found"))
            continue

        samples.append({
            "netlist_path": netlist_path, "rtl_path": rtl_path,
            "family": family, "variant": variant, "side": side,
        })

    if verbose:
        print(f"Matched {len(samples)} netlist+RTL pairs, skipped {len(skipped)} netlist files with no RTL match")
        if skipped:
            print("\nSkipped (family, variant, side, reason):")
            for row in skipped:
                print("  ", row)
    return samples, skipped


def report_dataset_coverage(root_dir):
    """Run before training: shows per-family sample counts and label balance
    so you can sanity check the data BEFORE burning GPU time on it."""
    samples, skipped = discover_samples(root_dir, verbose=True)

    by_family = defaultdict(lambda: {"total": 0, "y0": 0, "y1": 0})
    for s in samples:
        g = torch.load(s["netlist_path"], map_location="cpu", weights_only=False)
        y = g.y.item()
        by_family[s["family"]]["total"] += 1
        by_family[s["family"]][f"y{y}"] += 1

    print("\nPer-family usable sample counts (y0 = label 0, y1 = label 1):")
    for family, counts in by_family.items():
        print(f"  {family}: total={counts['total']}  y0={counts['y0']}  y1={counts['y1']}")

    total_variants = len(set((s["family"], s["variant"]) for s in samples))
    print(f"\nTotal usable samples: {len(samples)} across {total_variants} variants")
    return samples, skipped


class HWTrojanDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        graph = torch.load(s["netlist_path"], weights_only=False)
        rtl = torch.load(s["rtl_path"])
        return graph, rtl, graph.y.item()


samples, skipped = report_dataset_coverage("/content/HT_detection_GNN/trusthub/features/data")

Matched 83 netlist+RTL pairs, skipped 10 netlist files with no RTL match

Skipped (family, variant, side, reason):
   ('RS232', 'RS232-T100', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T200', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T300', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T400', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T500', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T600', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T700', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T800', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T900', 'standard', 'no matching RTL file found')
   ('RS232', 'RS232-T901', 'standard', 'no matching RTL file found')

Per-family usable sample counts (y0 = label 0, y1 = label 1):
  AES: total=53  y0=27  y1=26
  RS232: total=30  y0=4  y1=26

Total usable samples: 83 across 42 variants


In [20]:
import os

root = "/content/HT_detection_GNN/trusthub/features"
print("Top level:", os.listdir(root))

# show the first couple of subfolders, 2 levels deep
for name in os.listdir(root)[:3]:
    full = os.path.join(root, name)
    if os.path.isdir(full):
        print(f"\n{name}/")
        for sub in os.listdir(full)[:5]:
            print(f"  {sub}")

Top level: ['data', 'gate_vocab.json', 'manifest.json', 'metadata']

data/
  AES
  RS232
